In [ ]:
# Install needed libraries
%pip install -U python-jobspy
%pip install tqdm
%pip install xlsxwriter
%pip install tenacity requests

# Install MongoDB Python driver
%pip install pymongo
%pip install python-dotenv

In [ ]:
import sys
from pathlib import Path

# Get the absolute path of the project root (one level up from the notebooks directory)
project_root = str(Path().resolve().parent)  # Goes up two levels to reach the project root

# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from operations import (
    process_and_save_jobs, 
    setup_output_directory, 
    connect_to_mongodb,
    hours_old_since_2025,
    safe_scrape_jobs
)
import itertools

In [ ]:
connect_to_mongodb()

In [ ]:
# --- 1. Definir Directorio de Salida ---
output_dir = setup_output_directory("../data/raw")
print(f"Directorio de salida: {output_dir}")

## 2. Definir Parámetros de Búsqueda Base

In [ ]:
sectores_clave = ["Fintech", "EdTech", "Future of Work"]
search_terms = sectores_clave

#hours_old= hours_old_since_2025(2025)
hours_old = 24

indeed_glassdoor_countries = [
    "Australia",
    "Austria",
    "Belgium",
    "Brazil",
    "Canada",
    "France",
    "Germany",
    "Hong Kong",
    "India",
    "Ireland",
    "Italy",
    "Mexico",
    "Netherlands",
    "New Zealand",
    "Singapore",
    "Spain",
    "Switzerland",
    "UK",
    "USA",
    "Vietnam"
]

In [ ]:
# --- 3. Lista para guardar resultados ---
# Guardaremos los DataFrames de cada sitio aquí
all_jobs_dfs = []

In [ ]:
print("Parámetros listos. Iniciaremos scrapers secuenciales y especializados.")

In [ ]:
# --- 1. Scraper: Indeed (El "Caballo de batalla") ---

print("\n--- Iniciando Scraper: Indeed/Glassdoor ---")

for country_indeed, search_term in itertools.product(indeed_glassdoor_countries, search_terms):
    
    print(f"Buscando en {country_indeed} por {search_term} de hace {hours_old} horas")
    try:
        indeed_jobs = safe_scrape_jobs(
            site_name=["indeed", "glassdoor"],
            search_term=search_term,
            country_indeed=country_indeed,
            results_wanted=99999, # Considera bajar esto a 20-50 para pruebas rápidas
            hours_old=hours_old # Usamos el valor fijo calculado en la celda 6
        )
        
        if indeed_jobs is not None and not indeed_jobs.empty:
            print(f"✅ Se encontraron {len(indeed_jobs)} trabajos.")
            all_jobs_dfs.append(indeed_jobs)
            
    except Exception as e:
        print(f"❌ Error crítico en el bucle: {e}")

In [ ]:
"""print("\n--- Iniciando Scraper: Google ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        google_jobs = safe_scrape_jobs(
            site_name=["google"],
            search_term=search_term,
            google_search_term=f"{search_term}",
            results_wanted=10,
            hours_old=hours_old,
            verbose=2
        )
        if google_jobs is not None and not google_jobs.empty:
            print(f"✅ Se encontraron {len(google_jobs)} trabajos.")
            all_jobs_dfs.append(google_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

In [ ]:
"""print("\n--- Iniciando Scraper: ZipRecruiter ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        zip_jobs = safe_scrape_jobs(
            site_name=["zip_recruiter"],
            search_term=search_term,
            results_wanted=100, 
            hours_old=hours_old,
            verbose=2
        )
        if zip_jobs is not None and not zip_jobs.empty:
            print(f"✅ Se encontraron {len(zip_jobs)} trabajos.")
            all_jobs_dfs.append(zip_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

In [ ]:
"""print("\n--- Iniciando Scraper: Bayt, Naukri, BdJobs ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        bayt_jobs = safe_scrape_jobs(
            site_name=["bayt", "naukri", "bdjobs"],
            search_term=f"{search_term}",
            results_wanted=100, 
            hours_old=hours_old,
            verbose=2
        )
        if bayt_jobs is not None and not bayt_jobs.empty:
            print(f"✅ Se encontraron {len(bayt_jobs)} trabajos.")
            all_jobs_dfs.append(bayt_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

In [ ]:
print("\n--- Iniciando Scraper: Linkedin ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        linkedin_jobs = safe_scrape_jobs(
            site_name=["linkedin"],
            search_term=f"{search_term}",
            results_wanted=99999,
            linkedin_fetch_description=True,
            hours_old=hours_old,
            verbose=2
        )
        if linkedin_jobs is not None and not linkedin_jobs.empty:
            print(f"✅ Se encontraron {len(linkedin_jobs)} trabajos.")
            all_jobs_dfs.append(linkedin_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")

In [ ]:
print("\n--- Scraping secuencial completado ---")

In [ ]:
# Update your main processing loop:
if all_jobs_dfs:
    process_and_save_jobs(all_jobs_dfs, output_dir)
else:
    print("\nNo jobs were found.")